# Module 2 — Using Claude Code for String + File Processing

**SBML Lab Intern Training**

---

This module shows how Claude Code can generate file-processing and data-manipulation scripts from natural language descriptions.

The core skill is simple:

> **Describe what you want in plain English → get a script → refine it with follow-ups.**

By the end of this module you will have practiced this loop on GFF file parsing and pandas workflows — two tasks you will encounter constantly in the lab.


## Background: Genome Annotations and the GFF Format

### What is a genome annotation?

A genome is just a long string of DNA — billions of base pairs with no obvious structure by itself. A **genome annotation** is a map that labels where things are: this region is a gene, this region encodes a protein. Without an annotation, you have sequence but no biology.

The lab's annotation for *E. coli* K-12 MG1655 marks every gene in the genome — about 4,600 features total (all rows are `gene` features; this file has no separate `CDS` rows). This file is the reference you'll work with throughout the training.

### The GFF format

GFF (General Feature Format) is the standard file format for genome annotations. Every row describes one genomic feature (a gene, CDS, etc.) using 9 tab-separated columns:

The columns below describe the GFF format in general; the **Example** column shows the actual values found in this lab annotation file:

| Column | Name | Example (this file) |
|--------|------|---------|
| 1 | Chromosome / sequence name | `NC_000913` |
| 2 | Source | `Ecocyc_14.1` |
| 3 | Feature type | `gene` |
| 4 | Start position | `337` |
| 5 | End position | `2799` |
| 6 | Score | `.` (unused here) |
| 7 | Strand | `+` or `-` |
| 8 | Phase | `.` |
| 9 | Attributes | `locus_tag=b0002;gene=thrA;product=...;color=00FF00` |

> **Note the attribute keys:** this file uses `locus_tag=` and `gene=` (not `ID=`/`Name=`). Parse for those keys when extracting gene names.
>
> **Sequence-name convention:** this annotation uses `NC_000913` (no version suffix). The reference genome you download in Module 3 is `NC_000913.3`. The suffix differs — this matters when you join files by chromosome (Module 4), where `NC_000913` and `NC_000913.3` will not match unless you reconcile them.

**Coordinates are 1-based** — position 1 is the first nucleotide of the chromosome. This matters when writing Python code, since Python string indexing is 0-based. An off-by-one error here silently shifts every position by one.

### Why does this lab use GFF?

GFF is the input format for **MetaScope**, the lab's genome browser for visualizing ChIP-exo data. The alignment pipeline (Module 3) produces a GFF file as its final output so reads can be loaded into MetaScope alongside the gene annotation.

---

## GFF Format Recap

GFF is the standard annotation format used in this lab — tab-delimited, 9 columns, no header row, 1-based coordinates. Use `lab-context.md` as a reference if needed.

**Data for this module:** the lab *E. coli* K-12 MG1655 annotation GFF is pre-provided at:
```
data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
```
This is the same file used in Modules 4 and 5 (the mini-project).


## Natural Language → Script

The pattern you will use throughout this module:

1. **Write a plain-English description** of what you want the script to do. Be specific: mention the file format, the columns involved, and the expected output.
2. **Claude Code generates a first draft.** Paste it into a code cell and run it.
3. **Follow up with refinements.** Fix edge cases, add output formatting, handle errors, change the logic.

> **Important:** your first prompt is a draft, not a final answer. Follow-up prompts are where the real work happens. Most production scripts in the lab went through 3–10 iterations before they were committed.

### Tips for better first prompts

- Name the file format explicitly: *"a GFF file"*, not *"a text file"*.
- State the output: *"print the result to stdout"* or *"save to a new TSV file"*.
- Mention what to do with edge cases: *"skip comment lines"*, *"ignore rows where score is `.`"*.

---

### Why would you filter a genome annotation?

Real analyses rarely use the entire annotation. Common filtering scenarios in this lab:

- **By strand:** ChIP-exo reads are strand-specific — a binding site on the `+` strand regulates genes on the `+` strand. You often want to look at only one strand at a time.
- **By feature type:** You might want only `gene` entries, not `CDS`, depending on the question.
- **By coordinate range:** If you're zooming in on a genomic region (say, the first 500 kb), you only want features within those coordinates.

### What does "strand" mean?

The *E. coli* chromosome is double-stranded DNA. Genes can be encoded on either strand:

- **`+` strand (forward):** The gene is read left-to-right as written in the reference FASTA. Start < End.
- **`-` strand (reverse):** The gene is read right-to-left. In GFF, start is still less than end (by convention), but the strand column is `-`.

This matters biologically because a TF binding site upstream of a `+` strand gene is in a different genomic direction than one upstream of a `-` strand gene.

---

### Exercise 1 — Generate and refine a GFF filtering script

1. Describe a GFF filtering task in plain English to Claude Code. Use the lab annotation GFF at `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff`. For example: filter by feature type, by strand, or by coordinate range — your choice.
2. Generate the script.
3. Improve it with **at least two follow-up prompts**.

Write the **final version** of your script in the code cell below. Add a comment at the top of the script that briefly describes what each follow-up prompt changed.

Example comment format:
```python
# Follow-up 1: added argparse so the input path is a CLI argument
# Follow-up 2: changed output to TSV instead of printing to stdout
```

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [4]:
# Initial prompt: "load the GFF file at data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
#   and filter it to only the genes on the + strand"
# Follow-up 1: added a 'length' column (end - start) and sorted the forward-strand
#   genes by it, largest first.

import pandas as pd

GFF_PATH = '../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff'

df = pd.read_csv(GFF_PATH, sep='\t', header=None,
                  names=['seqid', 'source', 'feature', 'start', 'end',
                         'score', 'strand', 'phase', 'attributes'])

forward = df[df['strand'] == '+'].copy()
forward['length'] = forward['end'] - forward['start']+1
forward = forward.sort_values('length', ascending=False)
thrA = forward[forward['attributes'].str.contains('thrA')]


print(f'{len(forward)} of {len(df)} genes are on the + strand')
print(forward[['seqid', 'start', 'end', 'length', 'strand']].head())
print(thrA[['seqid', 'start', 'end', 'length', 'strand']].head())

# 불러올 data의 경로 지정 후 dataframe으로 불러오기. 이후 + strand에 해당하는 gene만 필터링하고, length 컬럼을 추가한 뒤 길이 기준으로 내림차순 정렬

2267 of 4595 genes are on the + strand
          seqid    start      end  length strand
2070  NC_000913  2042935  2050038    7104      +
1730  NC_000913  1727111  1731727    4617      +
3355  NC_000913  3352654  3357207    4554      +
956   NC_000913   975549   980009    4461      +
516   NC_000913   522485   526765    4281      +
       seqid  start   end  length strand
1  NC_000913    337  2799    2463      +


## Loading GFF Files with pandas

Loading a GFF file with pandas requires a few non-obvious options. Use Claude Code to figure out the correct `read_csv` call for `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff`.

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [5]:
import pandas as pd

GFF_PATH = '../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff'

df = pd.read_csv(GFF_PATH, sep='\t', header=None,
                  names=['seqid', 'source', 'feature', 'start', 'end',
                         'score', 'strand', 'phase', 'attributes'])

print(df.shape)
print(df.head())

#reference에 있는 데이터는 1행에 컬럼명이 없어서 헤더로 읽지 않도록 컬럼명을 직접 지정하여 csv파일로 변환하여 불러옴


(4595, 9)
       seqid       source feature  start   end score strand phase  \
0  NC_000913  Ecocyc_14.1    gene    190   255     .      +     .   
1  NC_000913  Ecocyc_14.1    gene    337  2799     .      +     .   
2  NC_000913  Ecocyc_14.1    gene   2801  3733     .      +     .   
3  NC_000913  Ecocyc_14.1    gene   3734  5020     .      +     .   
4  NC_000913  Ecocyc_14.1    gene   5234  5530     .      +     .   

                                          attributes  
0  locus_tag=b0001;gene=thrL;feature=gene;product...  
1  locus_tag=b0002;gene=thrA;feature=gene;product...  
2  locus_tag=b0003;gene=thrB;feature=gene;product...  
3  locus_tag=b0004;gene=thrC;feature=gene;product...  
4  locus_tag=b0005;gene=yaaX;feature=gene;product...  


### Exercise 2 — Debugging: Loading and Indexing

The script below has **two bugs**. Use Claude Code to diagnose and fix them.

Workflow:
1. Read the script and try to spot the bugs yourself first.
2. Paste the script into Claude Code and ask it to find the bugs.
3. Use `/debug` in the Claude Code terminal if you get stuck.
4. Write the corrected version in the empty code cell below the broken one.

> **Hint:** there is one bug in how the file is loaded, and one bug in how a row is accessed.

> **Explain it:** after fixing it, write 1–2 sentences on *what each bug was and why your fix works*. Understanding the bug matters more than the patch.

In [6]:
import pandas as pd

# Load GFF file
df = pd.read_csv('data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff', sep='\t')

# Filter for forward strand genes
forward = df[df['strand'] == '+']

# Get the 5th feature
fifth = forward.iat[5]

print(f"5th forward feature starts at position {fifth['start']}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff'

In [ ]:
# Fixed version here
import pandas as pd

# Load GFF file
df = pd.read_csv('../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff', sep='\t', header=None,
                  names=['seqid', 'source', 'feature', 'start', 'end',
                         'score', 'strand', 'phase', 'attributes'])

# Filter for forward strand genes
forward = df[df['strand'] == '+']

# Get the 5th feature
fifth = forward.iloc[4]

print(f"5th forward feature starts at position {fifth['start']}")

# 첫 번째 오류는 datafile을 불러올 때, 경로 지정이 잘못되어있고, header 설정과 컬럼명이 없어서 발생한 오류
# 두 번째 오류는 5번째 feature를 불러올 때, 행 전체를 가져오려면 .iat가 아닌 .iloc을 사용해야함. .iat는 (row,col) 두개의 인자를 필요로함
# 또한, 순서 지정이 0부터 시작하므로 5번째 feature를 가져오려면 4를 지정해야함

5th forward feature starts at position 5234


## `iterrows()` — When and Why

`iterrows()` iterates over a DataFrame row by row. Use Claude Code to understand when it is appropriate and when to avoid it.

### Exercise 3a — When to use it (concept)

After getting Claude Code's explanation, write a **2-sentence answer** in the markdown cell below:
- Sentence 1: when you **should** use `iterrows()`.
- Sentence 2: when you **should avoid** it and what to use instead.

### Exercise 3b — Use it on real data (hands-on)

Now actually iterate. Using the annotation DataFrame you loaded above, **find the gene whose transcription start site (TSS) is closest to position 1,000,000** on the genome.

- For a `+`-strand gene the TSS is its `start`; for a `-`-strand gene the TSS is its `end`.
- Loop over rows with `df.iterrows()`, compute each gene's distance to 1,000,000, and report the closest gene's name (the `gene=` value in the attribute column) and its distance.

Write the working script in the code cell below. When it runs, ask Claude Code how you would do the *same* computation **without** `iterrows()` (vectorized) — this is the practical version of the concept from Exercise 2a.


> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [ ]:
# Exercise 3b — find the gene whose TSS is closest to position 1,000,000, using df.iterrows()

import re

TARGET = 1_000_000

closest_gene = None
closest_distance = None

for idx, row in df.iterrows():
    tss = row['start'] if row['strand'] == '+' else row['end']
    distance = abs(tss - TARGET)
    if closest_distance is None or distance < closest_distance:
        closest_distance = distance
        match = re.search(r'gene=([^;]*)', row['attributes'])
        closest_gene = match.group(1) if match else None

print(f'Closest gene to position {TARGET}: {closest_gene} (distance: {closest_distance} bp)')

#strand가 +면 TSS는 start, -면 end이므로 조건이 행마다 다르므로 iterrows()를 사용
#abs(tss-TARGET)로 목표치와의 거리를 계산하고, 거리가 시행시 최솟값보다 작으면 closest_distance를 갱신하고, gene 이름을 attributes에서추출하여 closest_gene에 저장

Closest gene to position 1000000: ycbT (distance: 1030 bp)


> **Answer**
>
> *Your 2-sentence answer here.* 
>
> iterrows()는 행마다 조건이 달라서 벡터화된 연산이 불가능할 경우에 사용하는 방법이다. 하지만, 데이터의 개수가 많아질수록 연산 속도가 느려지므로, 대량의 데이터 연산 시 벡터화된 연산 방법을 고려해야한다.


## End of Session

Before closing, run `/log` in the Claude Code terminal to save a summary of your session.

This creates a record of the prompts you used and the scripts you generated — useful for your own reference and for lab documentation.

## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(module2): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [ ]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
